① Tool Definitions + Messages  
개발자가 사용할 수 있는 함수(예: get_weather(location))를 미리 정의  
사용자가 질문: “What’s the weather in Paris?”  
  
② Tool Calls  
모델이 질문을 보고 “아, 이건 get_weather("paris") 함수를 호출해야겠네”  
텍스트가 아니라 함수 호출 요청(JSON) 을 생성  
  
③ Execute Function Code  
실제 코드에서 get_weather("paris") 실행  
외부 API(OpenWeather 같은) 호출  
  
④ Results (All Prior Messages)  
함수 실행 결과가 다시 모델에게 전달  
모델은 이제 “파리의 온도 = 14도”라는 사실을 알게 됨  
  
⑤ Final Response  
모델이 사용자에게 자연어로 최종 답변 생성  
“It’s currently 14°C in Paris.”  
  
- LLM이 API를 직접 실행하는 게 아니라  
“어떤 함수를 호출할지 결정”만 하고  
실행은 개발자 코드,  
결과를 다시 받아 문장 생성  
👉 LLM + 외부 시스템 연동 구조  

In [1]:

from openai import OpenAI
from dotenv import load_dotenv
import os

load_dotenv()
client = OpenAI()
OPENWEATHER_API_KEY = os.getenv('OPENWEATHER_API_KEY')

## function(tool) 준비

In [ ]:
# OpenWeather API 호출해서 서울 날씨 데이터 조회
import requests

city_name = "Seoul"
units = "metric"
# OpenWeather 현재 날씨 API URL
url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}"
response = requests.get(url)
data = response.json() # 응답 JSON -> Python Dict
weather_info = {}

if response.status_code == 200:
    weather_description = data['weather'][0]['description']
    temp = data['main']['temp'] # 현재 온도
    temp_feels_like = data['main']['feels_like'] # 체감 온도
    humidity = data['main']['humidity'] # 습도

    weather_info = {
        'city': city_name,
        'description': weather_description,
        'temperature': temp,
        'temperature_feels_like': temp_feels_like,
        'humidity': humidity
    }
else:
    weather_info = {
        'city': city_name,
        'description': "Not Found",
        'temperature': "Not Found",
        'temperature_feels_like': "Not Found",
        'humidity': "Not Found"
    }

In [6]:
weather_info

{'city': 'Seoul',
 'description': 'overcast clouds',
 'temperature': 27.76,
 'temperature_feels_like': 32.84,
 'humidity': 89}

In [11]:
import json
def get_current_weather(city_name='Seoul', units='metric'):
    """
    OpenWeather 현재 날씨 API URL

    Args:
        - City: 도시 이름.(영문)
            - 서울 -> Seoul
            - 충청남도 -> Chungcheongnam-do
            - 부산 -> Busan
        - units: 온도단위 설정
            - metric (기본값: 섭씨, 미터)
            - imperial (화씨, 야드)
    Return:
        - str: json 형식으로 변환된 결과
    """
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city_name}&appid={OPENWEATHER_API_KEY}&units={units}"
    response = requests.get(url)
    data = response.json() # 응답 JSON -> Python Dict
    weather_info = {}

    if response.status_code == 200:
        weather_description = data['weather'][0]['description']
        temp = data['main']['temp'] # 현재 온도
        temp_feels_like = data['main']['feels_like'] # 체감 온도
        humidity = data['main']['humidity'] # 습도

        weather_info = {
            'city': city_name,
            'description': weather_description,
            'temperature': temp,
            'temperature_feels_like': temp_feels_like,
            'humidity': humidity
        }
    else:
        weather_info = {
            'city': city_name,
            'description': "Not Found",
            'temperature': "Not Found",
            'temperature_feels_like': "Not Found",
            'humidity': "Not Found"
        }
    return json.dumps(weather_info) # dict -> JSON

units: 온도단위 설정
- metric (기본값: 섭씨, 미터)
- imperial (화씨, 야드)

In [12]:
# llm이 사용할 함수 모음
tools_to_execute = {
    "get_current_weather" : get_current_weather # LLM이 호출할 tool 이름과 실제 함수 매핑
}

In [13]:
print(tools_to_execute['get_current_weather'].__doc__)


    OpenWeather 현재 날씨 API URL

    Args:
        - City: 도시 이름.(영문)
            - 서울 -> Seoul
            - 충청남도 -> Chungcheongnam-do
            - 부산 -> Busan
        - units: 온도단위 설정
            - metric (기본값: 섭씨, 미터)
            - imperial (화씨, 야드)
    Return:
        - str: json 형식으로 변환된 결과
    


In [ ]:
from pprint import pprint

# 사용자 질문을 받아서 tool 호출 여부를 판단하고 반영하여 최종 답변을 반환
def run_conversation(user_prompt, model='gpt-5.6-luna'):
    messages = [
        {'role': 'system', 'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수를 이용하여 필요한 정보를 먼저 확보한 후 대답해 주세요.'},
        {'role': 'user', 'content': user_prompt}
    ]
    tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_current_weather',
                'description': tools_to_execute['get_current_weather'].__doc__,
                'parameters': {
                    'type': 'object',
                    'properties': {
                        'city_name': {
                            'type': 'string',
                            'description': '''
                                도시 이름.(필수값, 영문)
                                -예시)
                                    - 서울 -> Seoul
                                    - 충청남도 -> Chungcheongnam-do
                                    - 부산 -> Busan
                            '''
                        },
                        'units':{
                            'type': 'string',
                            'description': '''
                                온도단위 설정
                                    - metric (기본값: 섭씨, 미터)
                                    - imperial (화씨, 야드)
                            ''',
                            'enum': ['metric', 'imperial']
                        }
                    },
                    'required': ['city_name']
                }
            }
        }
    ]

    # 첫 번째 LLM (함수 호출 필요 여부 판단)
    response1 = client.chat.completions.create(model = model, messages=messages, tools=tools, reasoning_effort='none')
    # reasoning_effort='none' -> 별도 추론없이 빠른 응답
    response1_message = response1.choices[0].message
    response1_tool_calls = response1_message.tool_calls

    if response1_tool_calls:
        messages.append(response1_message)

        for tool_call in response1_tool_calls:
            function_name = tool_call.function.name
            print(f'[tool] {function_name} 호출!')
            function_to_execute = tools_to_execute[function_name]
            function_args = json.loads(tool_call.function.arguments)
            function_response = function_to_execute(**function_args)

            messages.append({
                'role': 'tool',
                'tool_call_id': tool_call.id,
                'name': function_name,
                'content': function_response
            })
            pprint(messages)
        # 두 번째 LLM (tool 실행 결과를 포함한 히스토리로 결과를 자연스러운 언어로 출력)
        response2 = client.chat.completions.create(model = model, messages=messages)

        return response2.choices[0].message.content
    else:
        return response1_message.content

In [26]:
run_conversation('지금 서울날씨 어때?')


[tool] get_current_weather 호출!
[{'content': '당신은 친절한 챗봇입니다. 사용자의 요구를 분석해 직접 대답하거나, 주어진 함수를 이용하여 필요한 정보를 먼저 '
             '확보한 후 대답해 주세요.',
  'role': 'system'},
 {'content': '지금 서울날씨 어때?', 'role': 'user'},
 ChatCompletionMessage(content=None, refusal=None, role='assistant', annotations=[], audio=None, function_call=None, tool_calls=[ChatCompletionMessageFunctionToolCall(id='call_BNuMwy7aNrIbVhHJEEFVWuXD', function=Function(arguments='{"city_name":"Seoul","units":"metric"}', name='get_current_weather'), type='function')]),
 {'content': '{"city": "Seoul", "description": "overcast clouds", '
             '"temperature": 27.76, "temperature_feels_like": 33.64, '
             '"humidity": 94}',
  'name': 'get_current_weather',
  'role': 'tool',
  'tool_call_id': 'call_BNuMwy7aNrIbVhHJEEFVWuXD'}]


'현재 서울은 **흐리고**, 기온은 약 **27.8°C**입니다. 습도가 **94%**로 매우 높아 체감온도는 약 **33.6°C**로 후덥지근하게 느껴지겠습니다.  \n외출 시 수분 섭취에 유의하세요.'